# Building Optimization

This notebook performs building design optimization using EnergyPlus and bdopt.
We load a model from test.idf, define parameters to vary, set objectives, test the model, then run a multi-objective genetic algorithm (NSGA-II) and plot the optimized designs.

### Import libraries

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

from bdopt import (EnergyPlus, EnergyPlusEvaluator, FieldSelector, FunctionEvaluator, IntegerParameter,
                   Parameter, Problem, RangeParameter, load_idf, sampling, viz)
from bdopt.optimize import nsga2
from bdopt.surrogate import Surrogate
from window_to_wall import wwr      # the window-to-wall ratio as a design variable: see Besos_overview.ipynb

### Load the base EnergyPlus .idf file

In [ ]:
idf = load_idf("test.idf")
epw_file = "ITA_Torino.160590_IWEC.epw"

### Define design parameters and ranges
Define a parameter list, in this case the thickness of an insulation layer and the window-to-wall ratio.

(The previous toolbox built the first one with `expand_plist`; bdopt has no such short-cut, you write the class of the object. The window-to-wall ratio is a geometry edit written as a `GenericSelector` in `window_to_wall.py`, see `Besos_overview.ipynb`.)

In [ ]:
parameters = [
    Parameter(FieldSelector("Material", "XPS Extruded Polystyrene  - CO2 Blowing_0041", "Thickness"),
              RangeParameter(0.001, 0.3), name="Insulation Thickness"),      # [m]
    wwr(0.1, 0.8),                                                           # window-to-wall ratio between 0.1 and 0.8
]

### Objectives
Using Heating and Cooling energy outputs as simulation objectives, make a problem instance from these parameters and objectives.

In [ ]:
objectives = ["DistrictCooling:Facility", "DistrictHeating:Facility"]
problem = Problem(parameters, objectives)

### Set up the EnergyPlus evaluator object to run simulations for this building and problem

In [ ]:
evaluator = EnergyPlusEvaluator(
    problem, idf, epw_file,
    out_dir="/tmp/bdopt_runs",   # work files go to the container's own disk (the shared folder is much slower)
    err_dir="outputdir",         # runs that fail are kept here, with the EnergyPlus error file
    processes=2,                 # EnergyPlus runs in parallel
)
runs = pd.DataFrame({"Insulation Thickness": [0.2], "Window-to-wall ratio": [0.5]})   # one design: 0.2 m of insulation and 50% glazing

In [ ]:
runs

In [ ]:
outputs = evaluator.evaluate(runs)  # Run this as a test: one full-year EnergyPlus simulation
outputs

## Run the Genetic Algorithm
`nsga2` takes the problem and *what evaluates it*: here the EnergyPlus evaluator, so every design the algorithm tries is simulated (a full year each; each generation runs `pop_size` simulations, in parallel, and a design already simulated is taken from the cache).

To keep the lecture short we use a population of 10 designs for 3 generations (about 30 simulations); a real study needs much more. To plot every design that was tried, and not only the best ones, the evaluator is wrapped in a function that records what it computes: any Python function can play the role of an evaluator.

In [ ]:
tried = []      # every design the algorithm simulates, with its results

def simulate_and_record(designs):
    results = evaluator.evaluate(designs, keep_input=True)
    tried.append(results)
    return results[objectives].to_numpy()


recording_evaluator = FunctionEvaluator(problem, simulate_and_record, vectorized=True)

In [ ]:
front = nsga2(problem, recording_evaluator, pop_size=10, n_gen=3, seed=0)
results = pd.concat(tried, ignore_index=True).drop_duplicates()
front.df

### Visualize the results

In [ ]:
optres = front.df     # the non-dominated designs
plt.plot(
    results["DistrictCooling:Facility"], results["DistrictHeating:Facility"], "x"
)  # Plot all results in the background as blue crosses
plt.plot(
    optres["DistrictCooling:Facility"], optres["DistrictHeating:Facility"], "ro"
)  # Plot optimal results in red
plt.xlabel("Cooling demand")
plt.ylabel("Heating demand")

In [ ]:
optres = optres.sort_values("DistrictHeating:Facility")  # Sort by the first objective
ax = optres.plot.bar(
    subplots=True, legend=None, figsize=(10, 10)
)  # Plot the variable values and the objectives of each of the optimal solutions

### Giving constraints to the optimization

The previous version of this notebook defined here an evaluator with a constraint on a simulation *output* (`CO2:Facility <= 800`), but the next cell ran the optimisation with the evaluator *without* the constraint, so both runs were the same optimisation and the constraint was never used. That part is not in this notebook.

bdopt does not constrain simulation outputs. It constrains the *design variables*: `Problem(parameters, objectives, constraints=[g])`, where `g(row)` receives a design (a dictionary parameter name -> value) and returns a number; the design is feasible when the number is `<= 0`. Samplers only draw feasible designs and the optimizers respect the constraint. There is an example at the end of this notebook, with the surrogate model.

## Optimization with Surrogate Model

Simulating every design the algorithm tries is slow. A surrogate model trained on a few simulations answers in milliseconds, so the algorithm can try thousands of designs. `Surrogate_Modelling.ipynb` produced two files: the simulations (`surrogate_simple_simulation.csv`) and the trained surrogate (`best_surrogate.pkl`).

In [ ]:
# the simulations the surrogate was trained on: Surrogate_Modelling.ipynb must have been run first (in this same folder)
import os

missing = [f for f in ('surrogate_simple_simulation.csv', 'best_surrogate.pkl') if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(
        f"{', '.join(missing)} not found in {os.getcwd()}. These files are written by the last cells of Surrogate_Modelling.ipynb: "
        "open it, run it from the top to the end (it simulates 20 designs), then come back here.")
samples = pd.read_csv('surrogate_simple_simulation.csv', index_col=0)
samples

The problem is the one of `Surrogate_Modelling.ipynb`: five design variables and three objectives.

In [ ]:
parameters = [
    Parameter(FieldSelector('Material', 'XPS Extruded Polystyrene  - CO2 Blowing_0041', 'Thickness'),
              RangeParameter(0.01, 0.45), name='Insulation Thickness'),
    Parameter(FieldSelector('WindowMaterial:Glazing', '3', 'Conductivity'),
              RangeParameter(0.1, 1), name='Glazing conductivity'),
    Parameter(FieldSelector('WindowMaterial:Glazing', '3', 'Solar Transmittance at Normal Incidence'),
              RangeParameter(0.01, 0.8), name='Glazing solar transmittance'),
    wwr(0.1, 0.8),
    Parameter(FieldSelector('Lights', '*', 'Watts per Zone Floor Area'), IntegerParameter(8, 12), name='Lights Watts/Area'),
]
objectives = ['Electricity:Facility', 'DistrictHeating:Facility', 'DistrictCooling:Facility']   # EnergyPlus meters [J]
problem = Problem(parameters, objectives)

features = problem.parameter_names

# the table must have the columns of this problem (an old surrogate_simple_simulation.csv, for example the one that came
# with the course, has other names: RangeParameter [0.1, 0.9] ... for the window-to-wall ratio)
absent = [n for n in features + objectives if n not in samples.columns]
if absent:
    raise ValueError(
        f"surrogate_simple_simulation.csv has the columns {list(samples.columns)}, but this problem needs {absent} as well. "
        "The file is from another version of Surrogate_Modelling.ipynb: delete it and best_surrogate.pkl, then run "
        "Surrogate_Modelling.ipynb again from the top to the end, and come back here.")
inputs = samples[features]
outputs = samples[objectives]
inputs.head()

Load the surrogate. bdopt surrogates scale their inputs and outputs inside, so no scaler has to be rebuilt.

In [ ]:
try:
    surrogate = Surrogate.load('best_surrogate.pkl', problem)
    predicted = surrogate.predict(inputs)
    # the surrogate was trained on these very simulations, so it must be close to them (a file from another problem, or an old
    # scikit-learn pickle, loads without complaint and gives numbers that mean nothing)
    error = np.abs((np.asarray(predicted) - outputs.to_numpy()) / outputs.to_numpy()).mean()
    if not error < 0.5:
        raise ValueError(f"on the simulations it was trained on the surrogate is off by {100 * error:.0f}% on average")
except Exception as e:
    raise RuntimeError(
        f"best_surrogate.pkl does not fit this problem ({type(e).__name__}: {e}). It was probably saved by an older version of "
        "Surrogate_Modelling.ipynb or for a different problem: run Surrogate_Modelling.ipynb again from the top to the end.") from e
surrogate.predict(inputs.head(3))       # what the surrogate says about the first three simulated designs

In [ ]:
outputs.head(3)                         # what EnergyPlus said

In [ ]:
# running the NSGA-II optimizer on the surrogate: thousands of evaluations in seconds
front = nsga2(problem, surrogate, pop_size=100, n_gen=100, seed=0)

# computing Q: energy per square metre of floor area
HOUSE_DIMENSION = 118       # [m2]
results = front.df.copy()
results['Q'] = (results['Electricity:Facility'] + results['DistrictHeating:Facility'] + results['DistrictCooling:Facility']) / HOUSE_DIMENSION
results

In [ ]:
# plotting results: random designs evaluated by the surrogate in the background, the non-dominated designs in red
background = sampling.lhs(problem, 500, seed=1)
background = background.join(surrogate.predict(background))
viz.pareto(front, background=background)
plt.show()

In [ ]:
df = front.df.copy()
df['Total'] = df['Electricity:Facility'] + df['DistrictHeating:Facility'] + df['DistrictCooling:Facility']
df['Dist'] = np.sqrt(df['Electricity:Facility'] ** 2 + df['DistrictHeating:Facility'] ** 2 + df['DistrictCooling:Facility'] ** 2)

df[df.Dist == df.Dist.min()]

In [ ]:
viz.pairplot(front.df, problem, size=3)
plt.show()

In [ ]:
viz.correlation_heatmap(front.df, problem)
plt.show()

bdopt can also pick the design of the front closest to the ideal point, after scaling every objective to 0-1 over the front (so that an objective in joules does not weigh more only because of its numbers):

In [ ]:
front.closest_to_ideal()

### A constraint on a design variable

Suppose the insulation layer cannot be thicker than 0.25 m. The constraint is a function of the design: feasible when it returns a number `<= 0`.

In [ ]:
def thin_insulation(design):
    return design['Insulation Thickness'] - 0.25


constrained_problem = Problem(parameters, objectives, constraints=[thin_insulation])
front_constrained = nsga2(constrained_problem, surrogate, pop_size=100, n_gen=100, seed=0)

print('thickest insulation, without the constraint:', round(front.df['Insulation Thickness'].max(), 3), 'm')
print('thickest insulation, with the constraint:   ', round(front_constrained.df['Insulation Thickness'].max(), 3), 'm')

In [ ]:
plt.plot(front.df["DistrictCooling:Facility"], front.df["DistrictHeating:Facility"], "bo", label="no constraint")
plt.plot(front_constrained.df["DistrictCooling:Facility"], front_constrained.df["DistrictHeating:Facility"], "ro",
         label="insulation <= 0.25 m")
plt.xlabel("Cooling demand")
plt.ylabel("Heating demand")
plt.legend()
plt.show()